In [7]:
import numpy as np
import pandas as pd
N = 50000
np.random.seed(42)
col1 = np.random.standard_normal(N)
col2 = np.random.uniform(-5, 5, N)
col3, col4, col5 = np.random.multivariate_normal([1, -1, 2], [[1.0, 0.8, -0.3], [0.8, 1.0, 0.1], [-0.3, 0.1, 1.0]], N).reshape(3, -1)
col6 = np.random.rand(N)
col6 = np.random.choice(['Copper', *2*['Aluminium'], *3*['Titanium'], *4*['Tungsten']], N)
col7 = 2*col2**2 - 3*col2 + 1
col8rand = np.random.rand(N)
col8 = [np.random.normal(-3, 1) if rnd <= 0.7 else np.random.normal(3, 1) for rnd in col8rand]
col9 = [col1[i] * col2[i] if col6[i] != 'Copper' else None for i in range(N)]
# col9 = [col1[i] * col2[i] for i in range(N)]
df = pd.DataFrame({'col1': col1,'col2': col2,'col3': col3, 'col4': col4, 'col5': col5, 'col6': col6, 'col7': col7, 'col8': col8, 'col9': col9})
df

,col1,col2,col3,col4,col5,col6,col7,col8,col9
0,0.496714,-4.972295,0.856206,0.449057,-2.792878,Tungsten,65.364319,-3.982535,-2.469809
1,-0.138264,-1.174774,-1.104457,2.022651,0.897015,Tungsten,7.284507,-1.865070,0.162429
2,0.647689,4.140493,0.364906,-0.468112,0.758790,Aluminium,22.865882,-3.263226,2.681750
3,1.523030,4.477549,0.820167,2.298262,-1.647135,Copper,27.664247,2.511258,NaN
4,-0.234153,-4.464408,-0.931179,-0.346679,0.579473,Titanium,54.255091,-3.139977,1.045356
...,...,...,...,...,...,...,...,...,...
49995,0.056799,1.641270,1.613612,3.560890,-0.230378,Titanium,1.463726,-2.984232,0.093222
49996,-0.024923,-4.471092,-0.611411,0.536867,3.004438,Titanium,54.394594,-0.969403,0.111432
49997,0.500085,3.481545,1.098155,-2.128485,2.455626,Aluminium,14.797675,-2.538976,1.741068
49998,0.265215,4.167589,1.852977,1.266711,0.255052,Tungsten,23.234829,-2.824879,1.105309


In [8]:
y = pd.read_csv('train_data.csv')

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.impute import SimpleImputer
import optuna
df = pd.get_dummies(df)
# imputer = SimpleImputer()
# df = imputer.fit_transform(df)
train_x, val_x, train_y, val_y = train_test_split(df, y, test_size=0.05)
len(train_x), len(val_x)

(47500, 2500)

In [10]:
# sgd_model = SGDRegressor()
# sgd_model.fit(train_x, train_y)
# pred = sgd_model.predict(val_x)
# root_mean_squared_error(val_y, pred)

In [13]:
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 10)
    n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    lr = trial.suggest_float("lr", 1e-3, 0.3, log=True)
    subsample = trial.suggest_float("subsample", 0.6, 1.0)
    model = XGBRegressor(max_depth=max_depth, n_estimators=n_estimators, learning_rate=lr, subsample=subsample)
    # l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    # max_iter = trial.suggest_int('max_iter', 50, 1000)
    # model = SGDRegressor(l1_ratio=l1_ratio, max_iter=max_iter)
    model.fit(train_x, train_y)
    pred = model.predict(val_x)
    score = root_mean_squared_error(val_y, pred)
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10, show_progress_bar=True)
study.best_value, study.best_params

[I 2026-05-15 18:31:15,334] A new study created in memory with name: no-name-fa49516e-dde5-4e52-9a2a-b0d84e58fd43


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-05-15 18:31:15,956] Trial 0 finished with value: 6.602863311767578 and parameters: {'max_depth': 8, 'n_estimators': 218, 'lr': 0.10336354828804704, 'subsample': 0.9697276495453359}. Best is trial 0 with value: 6.602863311767578.
[I 2026-05-15 18:31:18,718] Trial 1 finished with value: 6.537807464599609 and parameters: {'max_depth': 9, 'n_estimators': 720, 'lr': 0.01364397620340565, 'subsample': 0.6113992691258363}. Best is trial 1 with value: 6.537807464599609.
[I 2026-05-15 18:31:18,808] Trial 2 finished with value: 6.620728492736816 and parameters: {'max_depth': 2, 'n_estimators': 51, 'lr': 0.11311682856478407, 'subsample': 0.6913255847545996}. Best is trial 1 with value: 6.537807464599609.
[I 2026-05-15 18:31:19,966] Trial 3 finished with value: 6.498423099517822 and parameters: {'max_depth': 8, 'n_estimators': 417, 'lr': 0.019346399371232965, 'subsample': 0.9197310772208379}. Best is trial 3 with value: 6.498423099517822.
[I 2026-05-15 18:31:21,141] Trial 4 finished with va

(6.458963871002197,
 {'max_depth': 5,
  'n_estimators': 862,
  'lr': 0.02028459090206698,
  'subsample': 0.7257932173648531})